# 02 - Data Preprocessing

- train on the past, test on the most recent month.

- Handle missing values (decide whether to drop, impute, or flag).
- Convert categorical fields to numeric (encoding).
- Normalize numerical features if needed.
- Create train/test split: use the most recent month of available data as the test set, and the X months immediately preceding it as the training set. X is not fixed — treat the training window length as a tunable choice and experiment to determine the optimal value of X.
-  Deliverable: 02_preprocessing.ipynb + cleaned CSV

In [169]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# ---------- load all files ----------
DATA_DIR = Path("../raw-data/csv/")

START, END = "2024-01", "2026-03"

months = pd.date_range(START, END, freq="MS").strftime("%Y%m")

frames, load_log = [], []
for ym in months:
    file_path = next(DATA_DIR.glob(f"CRMLSSold{ym}*.csv"), None)
    if file_path is None:
            print(f"Warning: no file found for {ym}")
            continue
    month_df = pd.read_csv(file_path, low_memory=False)
    month_df = month_df.drop(columns=["latfilled", "lonfilled"], errors="ignore")
    is_target_type = (month_df["PropertyType"] == "Residential") & (month_df["PropertySubType"] == "SingleFamilyResidence")
    month_df = month_df[is_target_type]
    # Add this month's data to the list
    frames.append(month_df)

s = pd.concat(frames, ignore_index=True) # merge all datframes

In [ ]:
# ---------- columns na percentage ----------
n0=len(s)
print(f"we initially had {n0} rows")
# drop na value rows from close date and close price (missing y value)
s = s.dropna(subset=["CloseDate", "ClosePrice"])
print("after filtering we have ", len(s), " rows left\n")

# find the na of other columns
miss = s.isna().mean().mul(100).round(2).sort_values(ascending=False)

we initially had 297245 rows
after filtering we have  297245  rows left



In [172]:
# drop empty and near empty columns (>95% nan)
cols_to_drop = miss[miss > 90].index

# Drop them from your dataframe
s = s.drop(columns=cols_to_drop)

# drop type and subtype
s=s.drop(columns=['PropertyType', 'PropertySubType'])

print(f"Dropped {len(cols_to_drop)} columns with >90% missing values.")
len(s.columns)

Dropped 17 columns with >90% missing values.


63

since we want to use a tree model and tree model can handle nan values well, we make the threathhold 95% for missing valeu columns

In [ ]:
# ---------- the remaining columns ----------
def profile(df, target="ClosePrice"):
    rows = []
    for c in df.columns:
        s = df[c]
        top = s.value_counts(normalize=True, dropna=True)
        rows.append({
            "col": c,
            "dtype": s.dtype,
            "missing_pct": round(s.isna().mean()*100, 1),
            "nunique": s.nunique(),
            "top_value_share": round(top.iloc[0], 3) if len(top) else None,
            "example": ", ".join(s.dropna().astype(str).head(3))
        })
    return pd.DataFrame(rows).set_index("col").sort_values("missing_pct", ascending=False)

#prof = profile(s)

In [174]:
df = s.copy()

In [ ]:
# ---------- dates / age ----------
for c in ["ListingContractDate", "CloseDate"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["list_year"] = df["ListingContractDate"].dt.year
df["list_month"] = df["ListingContractDate"].dt.month
df["house_age"] = df["list_year"] - df["YearBuilt"]

In [ ]:
# ---------- booleans -> 0/1 (keep NaN) ----------
yn_cols = []
for col in s.columns:
    vals = s[col].dropna().unique()
    
    if set(vals).issubset({True, False}):
        #print(f"{col}: {s[col].unique()}")
        yn_cols.append(col)
for c in yn_cols:
    df[c] = df[c].map({True: 1, False: 0, "True": 1, "False": 0}).astype("float")

In [ ]:
# ---------- association fee & house agency ----------
df["AssociationFee"] = df["AssociationFee"].fillna(0)
df["has_hoa"] = (df["AssociationFee"] > 0).astype(int) # hoa - house agency

In [ ]:
# ---------- lot size: keep sqft only ----------
df["LotSizeSquareFeet"] = df["LotSizeSquareFeet"].clip(upper=df["LotSizeSquareFeet"].quantile(0.995))

In [ ]:
# ---------- light feature engineering ----------
df["beds_baths"] = df["BedroomsTotal"] + df["BathroomsTotalInteger"]
df["sqft_per_bed"] = df["LivingArea"] / df["BedroomsTotal"].replace(0, np.nan)

In [ ]:
# ---------- drop columns ----------
leak = ["CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "DaysOnMarket",
        "BuyerAgencyCompensation", "BuyerAgencyCompensationType", "BuyerOfficeAOR",
        "BuyerOfficeName", "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId",
        "CoBuyerAgentFirstName"]
ids = ["ListingKey", "ListingKeyNumeric", "ListingId", "ListAgentEmail", "ListAgentFullName",
       "ListAgentFirstName", "ListAgentLastName", "ListOfficeName", "UnparsedAddress",
       "StreetNumberNumeric"]
const = ["MlsStatus", "PropertyType", "PropertySubType", "StateOrProvince"]
sparse = ["LotSizeDimensions", "CoListAgentFirstName",
          "CoListAgentLastName", "CoListOfficeName", "AssociationFeeFrequency", "SubdivisionName",
          "Flooring"]
dups = ["LotSizeAcres", "LotSizeArea"]

In [181]:
df = df.drop(columns=[c for c in leak + ids + const + sparse + dups if c in df.columns])

In [ ]:
# ---------- categoricals ----------
df.drop(columns=["PostalCode"])
cat_cols = ["City", "CountyOrParish", "MLSAreaMajor", "Levels"]
for c in cat_cols:
    df[c] = df[c].astype("string")
    # group rare categories
    vc = df[c].value_counts()
    rare = vc[vc < 20].index
    df.loc[df[c].isin(rare), c] = "Other"
    df[c] = df[c].astype("category") # Convert to Pandas categorical type (diff from string types)

In [183]:
#prof = profile(df)

In [184]:
df.to_csv("../data/clean_data.csv", index=False)